In [1]:
import pandas as pd
from Bio import SeqIO

In [2]:
!pwd

/var2/Works/junhyeong/RXLR_landscape/notebooks


In [3]:
%cd ../analysis/4.WY_tree/

/var2/Works/junhyeong/RXLR_landscape/analysis/4.WY_tree


/home/junhyeong/miniconda3/envs/biopython-env/lib/python3.13/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [4]:
!ls -al

total 8
drwxrwxr-x. 2 junhyeong junhyeong 4096 Dec 17 10:19 .
drwxrwxr-x. 6 junhyeong junhyeong 4096 Dec 17 10:19 ..


In [5]:
wy_prediction = pd.read_csv("../3.Clustering/3-1.domain_prediction/Predicted_WY_annotation_in_RXLR_effectors.tsv", sep = "\t")
pfam_prediction = pd.read_csv("../3.Clustering/3-1.domain_prediction/Predicted_Pfam_annotation_in_RXLR_effectors.tsv", sep = "\t")

In [6]:
structure_predicted = pd.read_csv("../3.Clustering/3-2.clustering/RXLR_cluster_domain.tsv", sep = "\t")

In [7]:
df = pd.merge(wy_prediction, structure_predicted, how = "right", on = ["entry", "spp"])

In [9]:
df = pd.merge(df, pfam_prediction, how = "left", on = ["entry", "spp"])

# Step 1

In [10]:
df["PFam"] = df["PFam"].fillna("<UNK>")

In [11]:
RXLR_set = ["<UNK>", "RXLR phytopathogen effector protein, Avirulence activity", "RXLR phytopathogen effector protein WY-domain", "WYL domain", "Avirulence protein ATR1, WY-domain", "Avirulence protein ATR13, RxLR effector"]

In [14]:
df = df[[pfam in RXLR_set for pfam in df["PFam"]]]

In [15]:
df

,entry,spp,WY,LWY,type,seq,seq_cl,str_cl,PFam
0,TDH65555.1,Brelac,0.0,0.0,no (L)WY,YGWRYPLRYWNRFGRPLFGSNCRFGRPWVGQLDVARCDDEKKEAEK...,-1,-1,<UNK>
1,TDH65737.1,Brelac,0.0,0.0,no (L)WY,YGWRYPLRYWNRKKERGEKEAFDVMTSSFMHIE,-1,-1,<UNK>
2,TDH66220.1,Brelac,0.0,0.0,no (L)WY,NPGIRHSMAWAFSEPINTVLRLKGITAEEAEAKLRAGKFYRKFLYG...,-1,-1,<UNK>
4,TDH66483.1,Brelac,0.0,0.0,no (L)WY,SLAGALASSLFGSSRSPQTVQSVDFQALLEPSPRLSNYDIKQRLRA...,-1,-1,<UNK>
5,TDH66629.1,Brelac,0.0,0.0,no (L)WY,ALVNLSEPLFYRAIRFLPLPKPDMAAIINLVKDLELSSSEALNNNL...,-1,-1,<UNK>
...,...,...,...,...,...,...,...,...,...
3956,OWY92363,Phymeg,0.0,0.0,no (L)WY,MPGGFYPTFIWKKKWEEHKKKLRRRSRSHKKRSSRTRAN,244,330,<UNK>
3957,OWZ11298,Phymeg,0.0,0.0,no (L)WY,NFDKFSKTHQKHLIRFAKNLGFDLKKSLTDTSYFAGISPDTIHKYH...,54,331,<UNK>
3958,OWZ06367,Phymeg,0.0,0.0,no (L)WY,GKGFFDFSREHQKQLKDFAKDLGFNLKAALKDPRVFNRIDEEAWEK...,54,331,<UNK>
3959,OWZ05828,Phymeg,0.0,0.0,no (L)WY,HFGNADDERKLRSDYSGYEEERHFGEP,396,332,<UNK>


# Step 2

In [16]:
wy_cl = {}
for wy, cl in zip(df["WY"].fillna(0), df["str_cl"]):
    if cl not in wy_cl:
        wy_cl[cl] = False
    if wy != 0:
        wy_cl[cl] = True

In [17]:
wy_member = []
seq_db = {}
for entry, seq, wy, cl in zip(df["entry"], df["seq"], df["WY"].fillna(0), df["str_cl"]):
    wy_contain = False
    if cl == -1:
        wy_contain = wy != 0
    else:
        wy_contain = wy_cl[cl]

    if wy_contain:
            
        seq_db[entry] = seq
    wy_member.append(wy_contain)

In [18]:
with open("RXLR_WY.fasta", "w") as f:
    for entry, seq in seq_db.items():
        f.write(f">{entry}\n{seq}\n")

In [19]:
df[wy_member].to_csv("RXLR_WY.tsv", sep = "\t", index = False)

In [22]:
rxlr_wy = df[wy_member]

In [25]:
import shutil

In [26]:
for entry in rxlr_wy["entry"]:
    shutil.copyfile(f"../2.Structure_prediction/pdb/{entry}.pdb", f"wy_pdb/{entry}.pdb")

In [27]:
rxlr_wy

,entry,spp,WY,LWY,type,seq,seq_cl,str_cl,PFam
9,TDH67508.1,Brelac,5.0,0.0,WY,MFPSGVGLKRTAGIPFETRPLKMSKSEADAVIKRIYHHANEHIDIA...,-1,-1,<UNK>
47,Hyaar1_13586,Hyaara,2.0,0.0,WY,TKDEAEREARMQPPSEISASTVAAKVKLNDIPGSNVATAIQTRLAQ...,-1,-1,<UNK>
74,Hyaar1_9367,Hyaara,2.0,0.0,WY,NAILETLSNHLASAFGYLWTKEAQEMTNSAWSALRNKESSLPLYSE...,-1,-1,<UNK>
158,Phyinf1_11083,Phyinf,1.0,0.0,WY,VVDDNERGFPRGPSLEKITETIESVAKKIWPKSKPSQSDVPDEKVS...,-1,-1,<UNK>
162,Phyinf1_13572,Phyinf,1.0,0.0,WY,GFNMWTTKAVPELLLADDQLQQLAKVSTSSDNVFKLLKLDDGLDGI...,-1,-1,<UNK>
...,...,...,...,...,...,...,...,...,...
3836,XP_009532482.1,Physoj,1.0,0.0,WY,AQPEGNDDDDSKPTDDDQKSKYETSADTKTKSDDPDEYTTAKSGKS...,15,270,<UNK>
3853,TDH69162.1,Brelac,0.0,0.0,no (L)WY,GKEERALWEEIAECFLQAQLGRGNIYRSLHIAEKMQNLKEQLLEAF...,-1,279,<UNK>
3854,TDH70024.1,Brelac,4.0,0.0,WY,YEKPARTDDEARTNIELEGLGNFVRNISHMGISNFLSRLIDESNPT...,279,279,<UNK>
3855,Phyinf1_13649,Phyinf,1.0,0.0,WY,GGTLRIPNQRIRKWLSKDLTPNKVKAKLSISSHTATDSEIYKLYQS...,332,280,<UNK>
